# Silver DQ: §6.1.2 metrics: unit tests + contract validation

**Run All.** The last cell prints PASS/FAIL per test and an N/M total.

Covers the ticket's four test bullets: unit per metric family · contract test
(`summary.json` + `metrics.json` validate against the committed JSON Schema) ·
per-MNO breakdown · idempotency. Plus an empty-table regression.

Fixtures are tiny in-memory frames: no Glue table is read, so this runs in about a
minute and can't break because a table was republished. The only external read is the
committed JSON Schema from S3, which the contract tests validate against.

In [1]:
%%pyspark project.spark.compatibility
import json, boto3, yaml, jsonschema
from datetime import datetime, date
from dataclasses import dataclass, field
from typing import Optional
from pyspark.sql import SparkSession, Column
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, LongType, StringType, StructField, StructType, TimestampType,
)

spark = (
    SparkSession.builder
    .appName("silver_dq_metrics_tests")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

SCHEMA_BASE = "s3://enstream-lake-silver-dev/poc/dq/schemas"   # published JSON Schema copies


Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility
Creating Glue session...
Create session for connection: project.spark.compatibility


'Session ae2xwmv6edkt56-e32409be-5025-40a7-bdf6-a8bbcd0aa0ed has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: project.spark.compatibility.

Connection: project.spark.compatibility | Run start time: 2026-07-22 00:46:40.842689 | Run duration : 0:02:30.990295s.


## Engine: copied verbatim from `silver_dq_run.ipynb`, keep in sync

Same four cells, same order. If you change an engine cell there, paste it here too: 
otherwise these tests pass against code that isn't the code you run.

In [2]:
%%pyspark project.spark.compatibility
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional


@dataclass(frozen=True)
class CheckConfig:
    """One silver table's DQ contract, parsed from that table's checks.yaml."""
    dataset: str
    primary_keys: list = field(default_factory=list)        # [] -> full-row
    partition_by: Optional[str] = None
    event_timestamp_col: str = "event_timestamp"

    timeliness_enabled: bool = False                        # set by presence of `timeliness:` block
    timeliness_mode: str = "batch"                          # batch = event-time | streaming = ingestion gap
    future_grace_seconds: int = 0                           # batch mode
    max_recency_seconds: Optional[int] = None               # batch mode, optional gate
    ingestion_timestamp_col: str = "ingestion_ts"           # streaming mode
    max_delay_seconds: Optional[int] = None                 # streaming mode, optional gate

    required_columns: list = field(default_factory=list)
    allowed_values: dict = field(default_factory=dict)              # {col:[vals]}
    allowed_values_by_mno: dict = field(default_factory=dict)       # {col:{mno:[vals]}}
    validity_formats: dict = field(default_factory=dict)            # {col: java-regex} (S12 Validity)
    victim_exclusion_col: Optional[str] = None                      # S12 Victim exclusion guard
    victim_allowed_values: list = field(default_factory=list)       #   values that mean "confirmed bad actor"
    currency_timestamp_col: str = "ingestion_ts"                    # S12 Currency: per-row age basis
    currency_max_age_seconds: Optional[int] = None                  #   review cycle; governance-provided

    def __post_init__(self):
        if not self.dataset:
            raise ValueError("checks.yaml must set 'dataset'")
        if self.allowed_values_by_mno and self.partition_by is None:
            raise ValueError("allowed_values_by_mno set but 'partition_by' (MNO col) is not")
        if self.timeliness_mode not in ("batch", "streaming"):
            raise ValueError(f"timeliness.mode must be 'batch' or 'streaming', got {self.timeliness_mode!r}")
        if (self.victim_exclusion_col is None) != (not self.victim_allowed_values):
            raise ValueError("victim_exclusion needs both 'column' and a non-empty 'allowed_values'")
        for col, pattern in (self.validity_formats or {}).items():
            if not isinstance(pattern, str) or not pattern:
                raise ValueError(f"validity.formats.{col} must be a non-empty regex string")
        if self.currency_max_age_seconds is not None and int(self.currency_max_age_seconds) <= 0:
            raise ValueError("timeliness.currency.max_age_seconds must be positive")


def parse_checks(raw: dict) -> CheckConfig:
    c = raw or {}
    completeness = c.get("completeness") or {}
    consistency  = c.get("consistency") or {}
    validity     = c.get("validity") or {}
    victim       = c.get("victim_exclusion") or {}
    tl           = c.get("timeliness") or {}
    currency     = tl.get("currency") or {}
    return CheckConfig(
        dataset=c.get("dataset"),
        primary_keys=list(c.get("primary_keys") or []),
        partition_by=c.get("partition_by"),
        event_timestamp_col=c.get("event_timestamp_col", "event_timestamp"),
        timeliness_enabled=("timeliness" in c),
        timeliness_mode=tl.get("mode", "batch"),
        future_grace_seconds=int(tl.get("future_grace_seconds", 0)),
        max_recency_seconds=tl.get("max_recency_seconds"),
        ingestion_timestamp_col=tl.get("ingestion_timestamp_col", "ingestion_ts"),
        max_delay_seconds=tl.get("max_delay_seconds"),
        required_columns=list(completeness.get("required_columns") or []),
        allowed_values=dict(consistency.get("allowed_values") or {}),
        allowed_values_by_mno=dict(consistency.get("allowed_values_by_mno") or {}),
        validity_formats=dict(validity.get("formats") or {}),
        victim_exclusion_col=victim.get("column"),
        victim_allowed_values=list(victim.get("allowed_values") or []),
        currency_timestamp_col=currency.get("timestamp_col", "ingestion_ts"),
        currency_max_age_seconds=currency.get("max_age_seconds"),
    )


def load_checks(path: str) -> CheckConfig:
    """Read checks.yaml from S3."""
    import boto3, yaml
    if not path.startswith("s3://"):
        raise ValueError(f"CHECKS_PATH must be an s3:// uri, got: {path}")
    bucket, key = path[len("s3://"):].split("/", 1)
    return parse_checks(yaml.safe_load(boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()))


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:11.838602 | Run duration : 0:00:00.818984s.


In [3]:
%%pyspark project.spark.compatibility
from pyspark.sql import Column
from pyspark.sql import functions as F

_NULL_SENTINEL = "\u0000NULL\u0000"
_KEY_SEP = "\u0001"


def alias_safe_name(column_name: str) -> str:
    """Strip characters that can't appear in an aggregation alias."""
    return column_name.replace(".", "_").replace("`", "")


def resolve_column(name, columns):
    """The table column matching `name` case-insensitively, or None.

    Same identifier rule Spark itself uses. Every family that takes a column name
    from the yaml resolves through here; see parse_completeness for the live
    incident that made case-blind matching mandatory.
    """
    fold = name.casefold()
    for c in columns:
        if c.casefold() == fold:
            return c
    return None


# completeness #
def completeness_exprs(columns):
    return [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"null_rows_{alias_safe_name(c)}")
            for c in columns]


def parse_completeness(row, columns, row_count, required):
    per_column, rules = {}, []
    # Yaml names must match table names the way Spark resolves columns: case-
    # insensitively. Proven live 2026-07-17 on account_changes_batch: Iceberg
    # preserves column case, the table has phone_number_AC_hash, the yaml says
    # phone_number_ac_hash a case-sensitive `c in required` produced 6 rules
    # from 7 required columns and the missing gate left no trace.
    required_by_fold = {r.casefold(): r for r in sorted(required)}
    matched_folds = set()
    for c in columns:
        nulls = int(row[f"null_rows_{alias_safe_name(c)}"] or 0)
        rate = round(nulls / row_count, 6) if row_count else 0.0
        per_column[c] = {"null_count": nulls, "null_rate": rate}
        if c.casefold() in required_by_fold:
            matched_folds.add(c.casefold())
            rules.append({"family": "completeness", "rule": f"{c} is not null",
                          "column": c, "violations": nulls, "passed": nulls == 0})
    # A required column the table lacks under ANY casing is a failed rule, not a
    # silent no-op otherwise a typo or a renamed column turns its gate off.
    for fold in sorted(set(required_by_fold) - matched_folds):
        name = required_by_fold[fold]
        rules.append({"family": "completeness", "rule": f"{name} exists in the table",
                      "column": name, "violations": 1, "passed": False})
    return {"per_column": per_column, "rules": rules}


# consistency #
def consistency_exprs(allowed_values, allowed_values_by_mno, mno_col):
    exprs = []
    for col, allowed in allowed_values.items():
        viol = F.col(col).isNotNull() & ~F.col(col).isin(list(allowed))
        exprs.append(F.sum(F.when(viol, 1).otherwise(0))
                     .alias(f"domain_violation_rows_{alias_safe_name(col)}"))
    for col, by_mno in allowed_values_by_mno.items():
        cond = F.lit(False)
        for mno, allowed in by_mno.items():
            cond = cond | ((F.col(mno_col) == mno) & F.col(col).isNotNull()
                           & ~F.col(col).isin(list(allowed)))
        exprs.append(F.sum(F.when(cond, 1).otherwise(0))
                     .alias(f"mno_domain_violation_rows_{alias_safe_name(col)}"))
    return exprs


def parse_consistency(row, allowed_values, allowed_values_by_mno, row_count):
    per_column, rules = {}, []
    for col, allowed in allowed_values.items():
        v = int(row[f"domain_violation_rows_{alias_safe_name(col)}"] or 0)
        per_column[col] = {"violation_count": v,
                           "violation_rate": round(v / row_count, 6) if row_count else 0.0,
                           "allowed_values": list(allowed)}
        rules.append({"family": "consistency", "rule": f"{col} in {sorted(map(str, allowed))}",
                      "column": col, "violations": v, "passed": v == 0})
    for col, by_mno in allowed_values_by_mno.items():
        v = int(row[f"mno_domain_violation_rows_{alias_safe_name(col)}"] or 0)
        per_column[col] = {"violation_count": v,
                           "violation_rate": round(v / row_count, 6) if row_count else 0.0,
                           "allowed_values_by_mno": {m: list(a) for m, a in by_mno.items()}}
        rules.append({"family": "consistency", "rule": f"{col} in per-MNO allowed set",
                      "column": col, "violations": v, "passed": v == 0})
    return {"per_column": per_column, "rules": rules}


# validity #
def validity_exprs(formats, columns):
    """One mismatch counter per configured format whose column exists (any casing).

    NULLs are completeness's job; a format check counts only non-null values that
    fail the pattern. Patterns are Java regex (Spark rlike) matched ANYWHERE in
    the value; anchor with ^...$ to require a full-string match.
    """
    exprs = []
    for name, pattern in sorted(formats.items()):
        c = resolve_column(name, columns)
        if c is not None:
            exprs.append(
                F.sum(F.when(F.col(c).isNotNull() & ~F.col(c).rlike(pattern), 1).otherwise(0))
                 .alias(f"format_violation_rows_{alias_safe_name(c)}"))
    return exprs


def parse_validity(row, formats, columns, row_count):
    per_column, rules = {}, []
    for name, pattern in sorted(formats.items()):
        c = resolve_column(name, columns)
        if c is None:
            # Fail closed, same as completeness: a format on a column the table
            # lacks under any casing must go red, not silently vanish.
            rules.append({"family": "validity", "rule": f"{name} exists in the table",
                          "column": name, "violations": 1, "passed": False})
            continue
        bad = int(row[f"format_violation_rows_{alias_safe_name(c)}"] or 0)
        per_column[c] = {"pattern": pattern, "violation_count": bad,
                         "violation_rate": round(bad / row_count, 6) if row_count else 0.0}
        rules.append({"family": "validity", "rule": f"{c} matches format",
                      "column": c, "violations": bad, "passed": bad == 0})
    return {"per_column": per_column, "rules": rules}


# uniqueness (null-safe key) #
def uniqueness_exprs(primary_keys, all_columns):
    keys = primary_keys or all_columns
    # coalesce each key part to a sentinel BEFORE concat (concat_ws drops nulls),
    # hash to fixed length, count distinct. A raw countDistinct over the key columns
    # would skip any row with a null key part -> wrong dedup rate on tables with
    # nullable keys (e.g. audit_trail full-row dedup where msisdn is 100% null).
    key = F.sha2(F.concat_ws(
        _KEY_SEP,
        *[F.coalesce(F.col(k).cast("string"), F.lit(_NULL_SENTINEL)) for k in keys],
    ), 256)
    return [F.countDistinct(key).alias("distinct_key_rows")]


def parse_uniqueness(row, primary_keys, row_count):
    distinct = int(row["distinct_key_rows"] or 0)
    dupes = row_count - distinct
    keys = primary_keys or ["<full row>"]
    return {"metrics": {"primary_keys": keys, "distinct_keys": distinct,
                        "duplicate_count": dupes,
                        "duplicate_rate": round(dupes / row_count, 6) if row_count else 0.0,
                        "distinct_rate": round(distinct / row_count, 6) if row_count else 0.0},
            "rules": [{"family": "uniqueness", "rule": f"unique on {keys}",
                       "column": ",".join(keys), "violations": dupes, "passed": dupes == 0}]}


# timeliness #
# Two modes, chosen by `timeliness.mode` in the yaml:
#
# batch event-time only. Gate = no event dated after the run. Use while the
# table is a one-shot historical backfill: every row's ingestion_ts is
# ~the same instant, so an ingestion gap measures "how old is this
# event" (0-3 years), not "how slow is the pipeline" no threshold
# is correct. Today all 4 tables are here, and 3 of them write
# ingestion_ts as an F.lit placeholder anyway.
#
# streaming ingestion gap = ingestion_ts - event_timestamp, the AC's definition. Gate =
# no row later than max_delay_seconds. Only meaningful once the table
# is fed incrementally AND ingestion_ts is stamped at write time.
#
# Both emit min/max_event_time so coverage is comparable across modes.
def timeliness_exprs(cfg, run_date, columns):
    ev = F.col(cfg.event_timestamp_col)
    exprs = [
        F.min(ev).cast("string").alias("min_event_time"),
        F.max(ev).cast("string").alias("max_event_time"),
        F.sum(F.when(ev.isNull(), 1).otherwise(0)).alias("null_event_time_rows"),
    ]
    if cfg.timeliness_mode == "streaming":
        ing = F.col(cfg.ingestion_timestamp_col)
        gap = F.unix_timestamp(ing) - F.unix_timestamp(ev)
        exprs += [
            F.min(ing).cast("string").alias("min_ingestion_time"),
            F.max(ing).cast("string").alias("max_ingestion_time"),
            F.max(gap).alias("max_gap_seconds"),
            F.sum(F.when(gap < 0, 1).otherwise(0)).alias("negative_gap_rows"),
            F.sum(F.when(ing.isNull(), 1).otherwise(0)).alias("null_ingestion_time_rows"),
            F.sum(F.when(F.lit(cfg.max_delay_seconds is not None)
                         & (gap > F.lit(cfg.max_delay_seconds or 0)), 1).otherwise(0))
             .alias("delay_violation_rows"),
        ]
    else:
        ev_unix = F.unix_timestamp(ev)
        as_of_unix = F.unix_timestamp(F.date_add(F.to_date(F.lit(run_date)), 1))  # midnight after run_date
        future_cutoff = as_of_unix + F.lit(int(cfg.future_grace_seconds))
        exprs += [
            F.min(as_of_unix - ev_unix).alias("recency_seconds"),   # == as_of - max(event_time)
            F.sum(F.when(ev_unix > future_cutoff, 1).otherwise(0)).alias("future_event_rows"),
        ]
    # S12 Currency: per-row review-cycle gate, valid in either mode. Age basis
    # defaults to ingestion_ts; as-of = midnight after run_date, the same instant
    # batch recency uses.
    if cfg.currency_max_age_seconds is not None:
        cur = resolve_column(cfg.currency_timestamp_col, columns)
        if cur is not None:
            cur_as_of = F.unix_timestamp(F.date_add(F.to_date(F.lit(run_date)), 1))
            age = cur_as_of - F.unix_timestamp(F.col(cur))
            exprs.append(
                F.sum(F.when(age > F.lit(int(cfg.currency_max_age_seconds)), 1).otherwise(0))
                 .alias("stale_rows"))
    return exprs


def parse_timeliness(row, cfg, columns, row_count):
    _rate = lambda n: round(n / row_count, 6) if row_count else 0.0
    metrics = {
        "mode": cfg.timeliness_mode,
        "event_timestamp_col": cfg.event_timestamp_col,
        "min_event_time": row["min_event_time"],
        "max_event_time": row["max_event_time"],
        "null_event_time_count": int(row["null_event_time_rows"] or 0),
    }
    if cfg.timeliness_mode == "streaming":
        delayed = int(row["delay_violation_rows"] or 0)
        negative = int(row["negative_gap_rows"] or 0)
        metrics.update({
            "ingestion_timestamp_col": cfg.ingestion_timestamp_col,
            "min_ingestion_time": row["min_ingestion_time"],
            "max_ingestion_time": row["max_ingestion_time"],
            "max_gap_seconds": int(row["max_gap_seconds"]) if row["max_gap_seconds"] is not None else None,
            "negative_gap_count": negative,
            "negative_gap_rate": _rate(negative),
            "null_ingestion_time_count": int(row["null_ingestion_time_rows"] or 0),
            "delay_violation_count": delayed,
            "delay_violation_rate": _rate(delayed),
            "max_delay_seconds": cfg.max_delay_seconds,
        })
        rules = [{"family": "timeliness", "rule": "no event_time after ingestion_time",
                  "column": cfg.event_timestamp_col, "violations": negative, "passed": negative == 0}]
        if cfg.max_delay_seconds is not None:
            rules.append({"family": "timeliness",
                          "rule": f"ingestion gap <= {cfg.max_delay_seconds}s",
                          "column": cfg.event_timestamp_col,
                          "violations": delayed, "passed": delayed == 0})
    else:
        future = int(row["future_event_rows"] or 0)
        recency = int(row["recency_seconds"]) if row["recency_seconds"] is not None else None
        metrics.update({
            "recency_seconds": recency,
            "future_count": future,
            "future_rate": _rate(future),
            "max_recency_seconds": cfg.max_recency_seconds,
        })
        rules = [{"family": "timeliness", "rule": "no future event_time",
                  "column": cfg.event_timestamp_col, "violations": future, "passed": future == 0}]
        if cfg.max_recency_seconds is not None:
            over = 1 if (recency is not None and recency > cfg.max_recency_seconds) else 0
            rules.append({"family": "timeliness", "rule": f"recency <= {cfg.max_recency_seconds}s",
                          "column": cfg.event_timestamp_col, "violations": over, "passed": over == 0})
    if cfg.currency_max_age_seconds is not None:
        cur = resolve_column(cfg.currency_timestamp_col, columns)
        if cur is None:
            rules.append({"family": "timeliness",
                          "rule": f"{cfg.currency_timestamp_col} exists in the table",
                          "column": cfg.currency_timestamp_col, "violations": 1, "passed": False})
        else:
            stale = int(row["stale_rows"] or 0)
            metrics["currency"] = {"timestamp_col": cur,
                                   "max_age_seconds": int(cfg.currency_max_age_seconds),
                                   "stale_count": stale, "stale_rate": _rate(stale)}
            rules.append({"family": "timeliness",
                          "rule": f"rows within review cycle ({cur} age <= {int(cfg.currency_max_age_seconds)}s)",
                          "column": cur, "violations": stale, "passed": stale == 0})
    return {"metrics": metrics, "rules": rules}


# victim exclusion #
def victim_exclusion_exprs(col_name, allowed, columns):
    """One counter: rows NOT attested as confirmed bad actor.

    NULL is a violation here, unlike consistency: an unattested record must not
    sit in the matchable population (governance framework S7.3 makes both
    attestations mandatory at ingestion), so a missing value fails the same as a
    wrong one.
    """
    c = resolve_column(col_name, columns)
    if c is None:
        return []
    return [F.sum(F.when(F.col(c).isin(list(allowed)), 0).otherwise(1))
             .alias("victim_violation_rows")]


def parse_victim_exclusion(row, col_name, allowed, columns, row_count):
    c = resolve_column(col_name, columns)
    if c is None:
        # Fail closed: a guard pointed at a column the table lacks is OFF, and an
        # off guard must be loud (see parse_completeness for the live incident).
        return {"metrics": {"column": col_name, "allowed_values": sorted(allowed),
                            "violation_count": None, "violation_rate": None},
                "rules": [{"family": "victim_exclusion",
                           "rule": f"{col_name} exists in the table",
                           "column": col_name, "violations": 1, "passed": False}]}
    bad = int(row["victim_violation_rows"] or 0)
    return {"metrics": {"column": c, "allowed_values": sorted(allowed),
                        "violation_count": bad,
                        "violation_rate": round(bad / row_count, 6) if row_count else 0.0},
            "rules": [{"family": "victim_exclusion",
                       "rule": f"{c} attests confirmed bad actor (nulls fail)",
                       "column": c, "violations": bad, "passed": bad == 0}]}


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:12.664485 | Run duration : 0:00:01.266123s.


In [4]:
%%pyspark project.spark.compatibility
SCHEMA_VERSION = "1.0.0"
FAMILIES = ("completeness", "consistency", "validity", "uniqueness", "timeliness", "victim_exclusion")


def _timeliness_on(cfg, columns):
    """Timeliness runs only if the yaml asked for it AND the columns it needs exist."""
    if not cfg.timeliness_enabled or cfg.event_timestamp_col not in columns:
        return False
    if cfg.timeliness_mode == "streaming":
        return cfg.ingestion_timestamp_col in columns
    return True


def _scan(df, cfg, columns, with_partition, run_date):
    """Read the table and return the raw per-family counts. THE ONLY SPARK STEP.

    Every family is concatenated into ONE .agg(), so a DQ run costs at most
    two passes over the table (one overall + one grouped by MNO) no matter how
    many rules the yaml configures.

    Do NOT split this per family or per output file. account_changes_batch is
    ~400M rows and audit_trail_services_3 ~436M; every extra split is another
    full pass. It would also let summary.json and metrics.json disagree: they are
    consistent today precisely because both are shaped from this one result.
    """
    exprs = [F.count(F.lit(1)).alias("row_count")]
    exprs += completeness_exprs(columns)
    exprs += consistency_exprs(cfg.allowed_values, cfg.allowed_values_by_mno, cfg.partition_by)
    exprs += validity_exprs(cfg.validity_formats, columns)
    exprs += uniqueness_exprs(cfg.primary_keys, columns)
    if cfg.victim_exclusion_col:
        exprs += victim_exclusion_exprs(cfg.victim_exclusion_col, cfg.victim_allowed_values, columns)
    if _timeliness_on(cfg, columns):
        exprs += timeliness_exprs(cfg, run_date, columns)
    if with_partition:
        return df.groupBy(cfg.partition_by).agg(*exprs).collect()
    return df.agg(*exprs).collect()


def _families_from_row(row, cfg, columns):
    """One _scan Row -> {row_count, <family>: {numbers, rules}}. Rates and verdicts happen here."""
    row_count = int(row["row_count"] or 0)
    out = {
        "row_count": row_count,
        "completeness": parse_completeness(row, columns, row_count, set(cfg.required_columns)),
        "consistency": parse_consistency(row, cfg.allowed_values, cfg.allowed_values_by_mno, row_count),
        "uniqueness": parse_uniqueness(row, cfg.primary_keys, row_count),
    }
    if cfg.validity_formats:
        out["validity"] = parse_validity(row, cfg.validity_formats, columns, row_count)
    if cfg.victim_exclusion_col:
        out["victim_exclusion"] = parse_victim_exclusion(
            row, cfg.victim_exclusion_col, cfg.victim_allowed_values, columns, row_count)
    if _timeliness_on(cfg, columns):
        out["timeliness"] = parse_timeliness(row, cfg, columns, row_count)
    return out


def _metrics_view(families):
    """Keep the numbers, drop the verdicts (verdicts are summary.json's job)."""
    out = {"row_count": families["row_count"]}
    for fam in FAMILIES:
        if fam in families:
            out[fam] = {k: v for k, v in families[fam].items() if k in ("per_column", "metrics")}
    return out


def build_summary(overall, cfg, run_date):
    """The verdicts -> summary.json. Pure Python: feed it a _families_from_row dict, no Spark."""
    rules = []
    for fam in FAMILIES:
        if fam in overall:
            rules.extend(overall[fam]["rules"])

    # An empty table satisfies every "violations == 0" rule vacuously: 0 nulls, 0 domain
    # violations, 0 duplicates, 0 future events -> passed=true for a table with no data.
    # That is the exact false green this module exists to prevent (see the ticket's
    # Motivation: lineage needs a dependable "is this silver run trustworthy" signal).
    # Verified 2026-07-16 on a real run: device_lookup_batch at row_count=0 reported
    # 16/16 rules passed and validated cleanly against the committed schema.
    if overall["row_count"] == 0:
        rules.append({"family": "completeness", "rule": "table is not empty",
                      "column": "*", "violations": 1, "passed": False})

    passed = sum(1 for r in rules if r["passed"])
    return {
        "schema_version": SCHEMA_VERSION,
        "dataset": cfg.dataset,
        "run_date": run_date,
        "row_count": overall["row_count"],
        "rule_count": len(rules),
        "passed_count": passed,
        "failed_count": len(rules) - passed,
        "passed": all(r["passed"] for r in rules),
        "rules": sorted(rules, key=lambda r: (r["family"], r["column"], r["rule"])),
    }


def build_metrics(overall, per_mno, cfg, run_date):
    """The numbers -> metrics.json. Pure Python: feed it _families_from_row dicts, no Spark."""
    return {
        "schema_version": SCHEMA_VERSION,
        "dataset": cfg.dataset,
        "run_date": run_date,
        "partition_by": cfg.partition_by,
        "overall": _metrics_view(overall),
        "per_mno": {k: _metrics_view(v) for k, v in per_mno.items()},
    }


def compute_dq(df, cfg, run_date):
    """Scan once, shape twice. Deterministic: reads only the DataFrame + cfg, never wall-clock."""
    columns = list(df.columns)
    overall = _families_from_row(_scan(df, cfg, columns, False, run_date)[0], cfg, columns)
    per_mno = {}
    if cfg.partition_by is not None:
        for r in _scan(df, cfg, columns, True, run_date):
            key = r[cfg.partition_by]
            per_mno[str(key) if key is not None else "__null__"] = _families_from_row(r, cfg, columns)
    return (build_summary(overall, cfg, run_date),
            build_metrics(overall, per_mno, cfg, run_date))


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:13.935887 | Run duration : 0:00:01.245236s.


In [5]:
%%pyspark project.spark.compatibility
import json, os


def result_paths(dq_root, cfg, run_date):
    base = f"{dq_root.rstrip('/')}/results/dataset={cfg.dataset}/run_date={run_date}"
    return f"{base}/summary.json", f"{base}/metrics.json"


def _dumps(obj):
    """The one serializer. The Inspect cells print through this too, so what you read
    in the notebook is byte-for-byte what write_results puts in S3."""
    return json.dumps(obj, sort_keys=True, indent=2, ensure_ascii=False)


def write_results(summary, metrics, dq_root, cfg, run_date):
    s_path, m_path = result_paths(dq_root, cfg, run_date)
    for path, obj in ((s_path, summary), (m_path, metrics)):
        payload = _dumps(obj)
        if path.startswith("s3://"):
            import boto3
            bucket, key = path[len("s3://"):].split("/", 1)
            boto3.client("s3").put_object(Bucket=bucket, Key=key,
                Body=payload.encode("utf-8"), ContentType="application/json")
        else:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            open(path, "w").write(payload)
    return s_path, m_path


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:15.185396 | Run duration : 0:00:00.759419s.


## Fixtures

In [6]:
%%pyspark project.spark.compatibility
# Fixtures. Tiny in-memory frames no Glue, no S3, so this notebook runs in ~1 min
# and can't be broken by a table being republished.

ACCT_SCHEMA = StructType([
    StructField("record_id", StringType()),
    StructField("phone_number_ac_hash", StringType()),
    StructField("event_type", StringType()),
    StructField("event_timestamp", TimestampType()),
    StructField("mno", StringType()),
    StructField("ingestion_ts", TimestampType()),
    StructField("source_name", StringType()),
    StructField("source_event_id", LongType()),
    StructField("schema_version", IntegerType()),
])

# rows 1 and 4 are identical -> one known duplicate; row 3 has a null hash.
ACCT_ROWS = [
    ("1", "h1", "SIM_CHANGE",       datetime(2026, 6, 20, 0, 0), "BELL",   datetime(2026, 6, 20, 1, 0), "MYSQL", None, 1),
    ("2", "h2", "DEVICE_CHANGE",    datetime(2026, 6, 20, 0, 0), "ROGERS", datetime(2026, 6, 20, 2, 0), "MYSQL", None, 1),
    ("3", None, "OWNERSHIP_CHANGE", datetime(2026, 6, 19, 0, 0), "TELUS",  datetime(2026, 6, 20, 0, 0), "MYSQL", None, 1),
    ("1", "h1", "SIM_CHANGE",       datetime(2026, 6, 20, 0, 0), "BELL",   datetime(2026, 6, 20, 1, 0), "MYSQL", None, 1),
]

ACCT_YAML = """
dataset: account_changes_batch
primary_keys: [record_id]
partition_by: mno
event_timestamp_col: event_timestamp
timeliness:
  mode: batch
  future_grace_seconds: 0
completeness:
  required_columns: [record_id, phone_number_ac_hash, event_type, event_timestamp, mno, source_name, schema_version]
consistency:
  allowed_values:
    mno: [BELL, ROGERS, TELUS]
  allowed_values_by_mno:
    event_type:
      BELL:   [SIM_CHANGE, DEVICE_CHANGE, OWNERSHIP_CHANGE]
      ROGERS: [SIM_CHANGE, DEVICE_CHANGE]
      TELUS:  [SIM_CHANGE, DEVICE_CHANGE, OWNERSHIP_CHANGE]
"""

def acct_df():
    return spark.createDataFrame(ACCT_ROWS, ACCT_SCHEMA)

def acct_cfg(**overrides):
    """Parse ACCT_YAML, optionally overriding top-level keys before parse.

    partition_by=None also drops allowed_values_by_mno: a per-MNO matrix with no MNO
    column is a contract error the config rightly rejects (see
    test_per_mno_allowed_values_needs_partition_by), so a caller asking for "no
    partition" means the flat config, not an invalid one.
    """
    raw = yaml.safe_load(ACCT_YAML)
    raw.update(overrides)
    if raw.get("partition_by") is None:
        raw.get("consistency", {}).pop("allowed_values_by_mno", None)
    return parse_checks(raw)

def s3_schema(name):
    """The committed JSON Schema, read from the published S3 copy; the same one
    silver_dq_run.ipynb validates against. Not a third inline copy that could drift."""
    bucket, key = f"{SCHEMA_BASE}/{name}.schema.json"[len("s3://"):].split("/", 1)
    return json.loads(boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"].read())

def validate(summary, metrics):
    jsonschema.validate(summary, s3_schema("summary"))
    jsonschema.validate(metrics, s3_schema("metrics"))

def rule_named(summary, text):
    return [r for r in summary["rules"] if r["rule"] == text]


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:15.950164 | Run duration : 0:00:01.253062s.


## Unit tests: one per metric family

In [7]:
%%pyspark project.spark.compatibility
# completeness
def test_completeness_counts_nulls_per_column():
    """Every column gets a null_count + null_rate, whether or not it's required."""
    cfg = acct_cfg()
    _, metrics = compute_dq(acct_df(), cfg, "2026-06-22")
    per_col = metrics["overall"]["completeness"]["per_column"]
    assert per_col["phone_number_ac_hash"]["null_count"] == 1, per_col["phone_number_ac_hash"]
    assert per_col["phone_number_ac_hash"]["null_rate"] == 0.25
    assert per_col["record_id"]["null_count"] == 0
    # source_event_id is 100% null but NOT required -> reported, never a rule
    assert per_col["source_event_id"]["null_rate"] == 1.0
    assert len(per_col) == 9, "all 9 columns must be reported, not just required ones"


def test_completeness_only_required_columns_become_rules():
    """required_columns drives the gate; the rest are KPI-only."""
    cfg = acct_cfg()
    summary, _ = compute_dq(acct_df(), cfg, "2026-06-22")
    cols = {r["column"] for r in summary["rules"] if r["family"] == "completeness"}
    assert cols == set(cfg.required_columns), cols
    assert "source_event_id" not in cols, "not required -> must not gate"
    assert rule_named(summary, "phone_number_ac_hash is not null")[0]["passed"] is False
    assert rule_named(summary, "record_id is not null")[0]["passed"] is True


def test_completeness_matches_required_case_insensitively():
    """Live find 2026-07-17: account_changes_batch has phone_number_AC_hash (Iceberg
    preserves column case) while the yaml says phone_number_ac_hash; 7 required
    columns produced only 6 rules and the missing gate left no trace."""
    sch = StructType([StructField("record_id", StringType()),
                      StructField("phone_number_AC_hash", StringType())])
    df = spark.createDataFrame([("1", "h1"), ("2", None)], sch)
    row = df.agg(*completeness_exprs(df.columns)).collect()[0]
    out = parse_completeness(row, df.columns, 2, {"phone_number_ac_hash"})
    assert len(out["rules"]) == 1, out["rules"]
    assert out["rules"][0]["column"] == "phone_number_AC_hash"
    assert out["rules"][0]["violations"] == 1 and out["rules"][0]["passed"] is False


def test_completeness_missing_required_column_goes_red():
    """A required column absent from the table (any casing) fails as a rule."""
    sch = StructType([StructField("record_id", StringType())])
    df = spark.createDataFrame([("1",)], sch)
    row = df.agg(*completeness_exprs(df.columns)).collect()[0]
    out = parse_completeness(row, df.columns, 1, {"record_id", "no_such_column"})
    missing = [r for r in out["rules"] if r["rule"] == "no_such_column exists in the table"]
    assert len(missing) == 1 and missing[0]["passed"] is False


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:17.208475 | Run duration : 0:00:00.784291s.


In [8]:
%%pyspark project.spark.compatibility
# consistency
def test_consistency_flat_allowed_values():
    """mno outside the allowed set is a violation; nulls are not (that's completeness)."""
    rows = ACCT_ROWS + [("9", "h9", "SIM_CHANGE", datetime(2026, 6, 20), "FREEDOM",
                         datetime(2026, 6, 20, 1), "MYSQL", None, 1)]
    df = spark.createDataFrame(rows, ACCT_SCHEMA)
    summary, metrics = compute_dq(df, acct_cfg(), "2026-06-22")
    assert metrics["overall"]["consistency"]["per_column"]["mno"]["violation_count"] == 1
    assert rule_named(summary, "mno in ['BELL', 'ROGERS', 'TELUS']")[0]["passed"] is False


def test_consistency_null_is_not_a_domain_violation():
    """A null mno is a completeness signal, not a consistency one."""
    rows = [("9", "h9", "SIM_CHANGE", datetime(2026, 6, 20), None,
             datetime(2026, 6, 20, 1), "MYSQL", None, 1)]
    df = spark.createDataFrame(rows, ACCT_SCHEMA)
    _, metrics = compute_dq(df, acct_cfg(partition_by=None), "2026-06-22")
    assert metrics["overall"]["consistency"]["per_column"]["mno"]["violation_count"] == 0
    assert metrics["overall"]["completeness"]["per_column"]["mno"]["null_count"] == 1


def test_consistency_per_mno_matrix():
    """OWNERSHIP_CHANGE is legal for BELL but not for ROGERS; same value, different verdict."""
    rows = [("1", "h1", "OWNERSHIP_CHANGE", datetime(2026, 6, 20), "BELL",
             datetime(2026, 6, 20, 1), "MYSQL", None, 1),
            ("2", "h2", "OWNERSHIP_CHANGE", datetime(2026, 6, 20), "ROGERS",
             datetime(2026, 6, 20, 1), "MYSQL", None, 1)]
    df = spark.createDataFrame(rows, ACCT_SCHEMA)
    _, metrics = compute_dq(df, acct_cfg(), "2026-06-22")
    assert metrics["overall"]["consistency"]["per_column"]["event_type"]["violation_count"] == 1


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:17.997061 | Run duration : 0:00:01.783243s.


In [9]:
%%pyspark project.spark.compatibility
# uniqueness
def test_uniqueness_duplicate_key_rate():
    """4 rows, record_id 1 twice -> 3 distinct, 1 duplicate."""
    summary, metrics = compute_dq(acct_df(), acct_cfg(), "2026-06-22")
    m = metrics["overall"]["uniqueness"]["metrics"]
    assert m["distinct_keys"] == 3, m
    assert m["duplicate_count"] == 1
    assert m["duplicate_rate"] == 0.25
    assert m["distinct_rate"] == 0.75, "AC asks metrics.json for distinct %"
    assert m["distinct_rate"] + m["duplicate_rate"] == 1.0
    assert rule_named(summary, "unique on ['record_id']")[0]["passed"] is False


def test_uniqueness_null_key_parts_still_counted():
    """concat_ws DROPS nulls, so a raw countDistinct would collapse ('1', null) and
    ('1',) into one key and under-report duplicates. The null sentinel prevents that."""
    rows = [("1", None, "SIM_CHANGE", datetime(2026, 6, 20), "BELL", datetime(2026, 6, 20, 1), "MYSQL", None, 1),
            ("1", "h1", "SIM_CHANGE", datetime(2026, 6, 20), "BELL", datetime(2026, 6, 20, 1), "MYSQL", None, 1)]
    df = spark.createDataFrame(rows, ACCT_SCHEMA)
    cfg = acct_cfg(primary_keys=["record_id", "phone_number_ac_hash"])
    _, metrics = compute_dq(df, cfg, "2026-06-22")
    m = metrics["overall"]["uniqueness"]["metrics"]
    assert m["distinct_keys"] == 2, f"null key part must stay distinct, got {m}"
    assert m["duplicate_count"] == 0


def test_uniqueness_no_primary_keys_means_full_row():
    cfg = acct_cfg(primary_keys=[])
    summary, metrics = compute_dq(acct_df(), cfg, "2026-06-22")
    assert metrics["overall"]["uniqueness"]["metrics"]["primary_keys"] == ["<full row>"]
    assert metrics["overall"]["uniqueness"]["metrics"]["duplicate_count"] == 1


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:19.785645 | Run duration : 0:00:00.241075s.


In [10]:
%%pyspark project.spark.compatibility
# timeliness
def test_timeliness_batch_future_event_fails():
    """An event dated after the run is a clock/parse bug -> the batch-mode gate."""
    rows = [("1", "h1", "SIM_CHANGE", datetime(2026, 6, 25), "BELL", datetime(2026, 6, 20, 1), "MYSQL", None, 1)]
    df = spark.createDataFrame(rows, ACCT_SCHEMA)
    summary, metrics = compute_dq(df, acct_cfg(), "2026-06-22")
    m = metrics["overall"]["timeliness"]["metrics"]
    assert m["mode"] == "batch"
    assert m["future_count"] == 1, m
    assert rule_named(summary, "no future event_time")[0]["passed"] is False


def test_timeliness_batch_coverage_window_and_recency():
    """min/max_event_time bound the data; recency = (midnight after run_date) - max(event)."""
    _, metrics = compute_dq(acct_df(), acct_cfg(), "2026-06-22")
    m = metrics["overall"]["timeliness"]["metrics"]
    assert m["min_event_time"].startswith("2026-06-19")
    assert m["max_event_time"].startswith("2026-06-20")
    assert m["future_count"] == 0
    # max event = 2026-06-20 00:00; as_of = 2026-06-23 00:00 -> 3 days
    assert m["recency_seconds"] == 3 * 86400, m["recency_seconds"]


def test_timeliness_streaming_gap_and_delay_gate():
    """mode: streaming implements the ticket's definition; ingestion_ts - event_timestamp."""
    rows = [("1", "h1", "SIM_CHANGE", datetime(2026, 6, 1), "BELL",
             datetime(2026, 6, 20), "MYSQL", None, 1)]          # 19-day gap
    df = spark.createDataFrame(rows, ACCT_SCHEMA)
    raw = yaml.safe_load(ACCT_YAML)
    raw["timeliness"] = {"mode": "streaming", "ingestion_timestamp_col": "ingestion_ts",
                         "max_delay_seconds": 604800}            # 7 days
    summary, metrics = compute_dq(df, parse_checks(raw), "2026-06-22")
    m = metrics["overall"]["timeliness"]["metrics"]
    assert m["mode"] == "streaming"
    assert m["max_gap_seconds"] == 19 * 86400, m["max_gap_seconds"]
    assert m["delay_violation_count"] == 1
    assert rule_named(summary, "ingestion gap <= 604800s")[0]["passed"] is False
    assert not rule_named(summary, "no future event_time"), "batch-only rule leaked into streaming"


def test_timeliness_streaming_negative_gap_is_a_clock_bug():
    """event_timestamp after ingestion_ts means the row was ingested before it happened."""
    rows = [("1", "h1", "SIM_CHANGE", datetime(2026, 6, 20), "BELL",
             datetime(2026, 6, 1), "MYSQL", None, 1)]
    df = spark.createDataFrame(rows, ACCT_SCHEMA)
    raw = yaml.safe_load(ACCT_YAML)
    raw["timeliness"] = {"mode": "streaming", "ingestion_timestamp_col": "ingestion_ts"}
    summary, metrics = compute_dq(df, parse_checks(raw), "2026-06-22")
    assert metrics["overall"]["timeliness"]["metrics"]["negative_gap_count"] == 1
    assert rule_named(summary, "no event_time after ingestion_time")[0]["passed"] is False


def test_timeliness_absent_when_yaml_omits_it():
    raw = yaml.safe_load(ACCT_YAML)
    del raw["timeliness"]
    _, metrics = compute_dq(acct_df(), parse_checks(raw), "2026-06-22")
    assert "timeliness" not in metrics["overall"], "no timeliness: block -> family must not run"


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:20.031151 | Run duration : 0:00:02.205061s.


## Empty table: regression

A real 2026-07-15 run of `device_lookup_batch` at `row_count=0` reported 16/16 rules
passed and validated cleanly, and that green `summary.json` was written to S3. An
empty table satisfies every "violations == 0" rule vacuously.

In [11]:
%%pyspark project.spark.compatibility
# empty table
# Regression for a real 2026-07-15 run: device_lookup_batch at row_count=0 reported
# 16/16 rules passed and validated cleanly, and that green summary was written to
# poc/dq/results/dataset=device_lookup_batch/run_date=2026-07-15/.
def test_empty_table_must_not_report_passed():
    summary, _ = compute_dq(acct_df().limit(0), acct_cfg(), "2026-07-16")
    assert summary["row_count"] == 0
    assert summary["passed"] is False, "empty table reported passed=true; the false green is back"
    empty = rule_named(summary, "table is not empty")
    assert len(empty) == 1 and empty[0]["passed"] is False and empty[0]["violations"] == 1


def test_empty_table_verdict_is_still_schema_valid():
    """The red verdict must be a contract-valid artifact, not a crash."""
    summary, metrics = compute_dq(acct_df().limit(0), acct_cfg(), "2026-07-16")
    validate(summary, metrics)


def test_non_empty_table_has_no_empty_rule():
    summary, _ = compute_dq(acct_df(), acct_cfg(), "2026-07-16")
    assert summary["row_count"] == 4
    assert not rule_named(summary, "table is not empty"), "guard fired on a table with rows"


def test_build_summary_empty_needs_no_spark():
    """build_summary is pure Python; feed it a dict, no DataFrame, no SparkSession."""
    families = {"row_count": 0, "uniqueness": {"metrics": {}, "rules": []}}
    summary = build_summary(families, parse_checks({"dataset": "demo"}), "2026-07-16")
    assert summary["passed"] is False and summary["rule_count"] == 1


Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:22.240840 | Run duration : 0:00:00.754689s.


## Contract · per-MNO · idempotency

The three remaining test bullets from the ticket, plus config-validation checks.

In [12]:
%%pyspark project.spark.compatibility
# contract / per-MNO / idempotency
def test_outputs_validate_against_committed_schema():
    """AC: summary.json + metrics.json validate against the committed JSON Schema.
    Reads the published S3 copy; the same file silver_dq_run.ipynb validates against."""
    summary, metrics = compute_dq(acct_df(), acct_cfg(), "2026-06-22")
    validate(summary, metrics)


def test_metrics_carries_no_verdicts_and_summary_no_numbers():
    """The two files split by role: rules -> summary, numbers -> metrics. No overlap."""
    summary, metrics = compute_dq(acct_df(), acct_cfg(), "2026-06-22")
    for fam, block in metrics["overall"].items():
        if isinstance(block, dict):
            assert "rules" not in block, f"{fam}: verdicts leaked into metrics.json"
    assert "per_column" not in summary, "numbers leaked into summary.json"
    assert summary["rule_count"] == len(summary["rules"])
    assert summary["passed_count"] + summary["failed_count"] == summary["rule_count"]


def test_per_mno_breakdown_in_addition_to_overall():
    """AC: a mixed-MNO fixture produces per-MNO metrics IN ADDITION TO overall."""
    _, metrics = compute_dq(acct_df(), acct_cfg(), "2026-06-22")
    assert set(metrics["per_mno"]) == {"BELL", "ROGERS", "TELUS"}, list(metrics["per_mno"])
    assert metrics["per_mno"]["BELL"]["row_count"] == 2       # the duplicate pair
    assert metrics["per_mno"]["ROGERS"]["row_count"] == 1
    assert metrics["overall"]["row_count"] == 4
    # BELL's two rows are identical -> 1 distinct key inside that group
    assert metrics["per_mno"]["BELL"]["uniqueness"]["metrics"]["duplicate_count"] == 1


def test_per_mno_distincts_do_not_sum_to_overall():
    """Why the overall pass can't be derived from the per-MNO pass: a key spanning two
    MNOs is distinct in each group but a duplicate overall. Locks in the 2-scan design."""
    rows = [("1", "h1", "SIM_CHANGE", datetime(2026, 6, 20), "BELL",   datetime(2026, 6, 20, 1), "MYSQL", None, 1),
            ("1", "h1", "SIM_CHANGE", datetime(2026, 6, 20), "ROGERS", datetime(2026, 6, 20, 1), "MYSQL", None, 1)]
    df = spark.createDataFrame(rows, ACCT_SCHEMA)
    _, metrics = compute_dq(df, acct_cfg(), "2026-06-22")
    per_mno_sum = sum(m["uniqueness"]["metrics"]["distinct_keys"] for m in metrics["per_mno"].values())
    overall = metrics["overall"]["uniqueness"]["metrics"]["distinct_keys"]
    assert per_mno_sum == 2 and overall == 1, f"{per_mno_sum} vs {overall}"


def test_no_partition_by_means_no_per_mno_scan():
    _, metrics = compute_dq(acct_df(), acct_cfg(partition_by=None), "2026-06-22")
    assert metrics["per_mno"] == {}
    assert metrics["partition_by"] is None


def test_idempotent_rerun_is_byte_identical():
    """AC: rerunning DQ against the same snapshot produces identical outputs."""
    df, cfg = acct_df(), acct_cfg()
    a_s, a_m = compute_dq(df, cfg, "2026-06-22")
    b_s, b_m = compute_dq(df, cfg, "2026-06-22")
    assert _dumps(a_s) == _dumps(b_s), "summary.json not reproducible"
    assert _dumps(a_m) == _dumps(b_m), "metrics.json not reproducible"


def test_result_paths_are_canonical():
    cfg = acct_cfg()
    s_path, m_path = result_paths("s3://enstream-lake-silver-dev/poc/dq", cfg, "2026-06-22")
    assert s_path == ("s3://enstream-lake-silver-dev/poc/dq/results/"
                      "dataset=account_changes_batch/run_date=2026-06-22/summary.json"), s_path
    assert m_path.endswith("/metrics.json")


def test_config_rejects_bad_mode_and_missing_dataset():
    for bad in ({"dataset": "d", "timeliness": {"mode": "realtime"}}, {"dataset": None}):
        try:
            parse_checks(bad)
        except ValueError:
            continue
        raise AssertionError(f"should have raised: {bad}")


def test_per_mno_allowed_values_needs_partition_by():
    try:
        parse_checks({"dataset": "d", "consistency": {"allowed_values_by_mno": {"event_type": {"BELL": ["X"]}}}})
    except ValueError:
        return
    raise AssertionError("allowed_values_by_mno without partition_by should raise")


# S12 additions: validity / victim exclusion / currency #
def _gold_frame(rows):
    from pyspark.sql.types import StringType, StructField, StructType, TimestampType
    schema = StructType([StructField("record_id", StringType()),
                         StructField("victim_attestation", StringType()),
                         StructField("phone_number", StringType()),
                         StructField("ingestion_ts", TimestampType())])
    return spark.createDataFrame(rows, schema)


def test_validity_counts_non_null_format_mismatches():
    from datetime import datetime
    df = _gold_frame([("1", "CONFIRMED_BAD_ACTOR", "+14165551234", datetime(2026, 7, 10)),
                      ("2", "CONFIRMED_BAD_ACTOR", "abc",          datetime(2026, 7, 10)),
                      ("3", "CONFIRMED_BAD_ACTOR", None,           datetime(2026, 7, 10))])
    cfg = parse_checks({"dataset": "g", "primary_keys": ["record_id"],
                        "validity": {"formats": {"PHONE_NUMBER": r"^\+1[0-9]{10}$"}}})
    summary, metrics = compute_dq(df, cfg, "2026-07-16")
    assert metrics["overall"]["validity"]["per_column"]["phone_number"]["violation_count"] == 1
    rule = next(r for r in summary["rules"] if r["family"] == "validity")
    assert rule["rule"] == "phone_number matches format" and rule["passed"] is False


def test_victim_exclusion_counts_wrong_and_null():
    from datetime import datetime
    df = _gold_frame([("1", "CONFIRMED_BAD_ACTOR", "+14165551234", datetime(2026, 7, 10)),
                      ("2", "VICTIM",              "+14165551235", datetime(2026, 7, 10)),
                      ("3", None,                  "+14165551236", datetime(2026, 7, 10))])
    cfg = parse_checks({"dataset": "g", "primary_keys": ["record_id"],
                        "victim_exclusion": {"column": "VICTIM_ATTESTATION",
                                             "allowed_values": ["CONFIRMED_BAD_ACTOR"]}})
    summary, metrics = compute_dq(df, cfg, "2026-07-16")
    vx = metrics["overall"]["victim_exclusion"]["metrics"]
    assert vx["column"] == "victim_attestation" and vx["violation_count"] == 2
    assert next(r for r in summary["rules"] if r["family"] == "victim_exclusion")["passed"] is False


def test_currency_counts_rows_past_review_cycle():
    from datetime import datetime
    df = _gold_frame([("1", "CONFIRMED_BAD_ACTOR", "+14165551234", datetime(2026, 7, 10)),
                      ("2", "CONFIRMED_BAD_ACTOR", "+14165551235", datetime(2026, 7, 12)),
                      ("3", "CONFIRMED_BAD_ACTOR", "+14165551236", datetime(2026, 5, 1))])
    cfg = parse_checks({"dataset": "g", "primary_keys": ["record_id"],
                        "event_timestamp_col": "ingestion_ts",
                        "timeliness": {"mode": "batch",
                                       "currency": {"timestamp_col": "ingestion_ts",
                                                    "max_age_seconds": 30 * 86400}}})
    summary, metrics = compute_dq(df, cfg, "2026-07-16")
    assert metrics["overall"]["timeliness"]["metrics"]["currency"]["stale_count"] == 1
    rule = next(r for r in summary["rules"] if "review cycle" in r["rule"])
    assert rule["violations"] == 1 and rule["passed"] is False



Connection: project.spark.compatibility | Run start time: 2026-07-22 00:49:23.001209 | Run duration : 0:00:02.231198s.


## Run

In [13]:
%%pyspark project.spark.compatibility
# Run every test. Convention matches TU_PortPS_silver_tests.ipynb.
_tests = [
    test_completeness_counts_nulls_per_column,
    test_completeness_only_required_columns_become_rules,
    test_completeness_matches_required_case_insensitively,
    test_completeness_missing_required_column_goes_red,
    test_consistency_flat_allowed_values,
    test_consistency_null_is_not_a_domain_violation,
    test_consistency_per_mno_matrix,
    test_uniqueness_duplicate_key_rate,
    test_uniqueness_null_key_parts_still_counted,
    test_uniqueness_no_primary_keys_means_full_row,
    test_timeliness_batch_future_event_fails,
    test_timeliness_batch_coverage_window_and_recency,
    test_timeliness_streaming_gap_and_delay_gate,
    test_timeliness_streaming_negative_gap_is_a_clock_bug,
    test_timeliness_absent_when_yaml_omits_it,
    test_empty_table_must_not_report_passed,
    test_empty_table_verdict_is_still_schema_valid,
    test_non_empty_table_has_no_empty_rule,
    test_build_summary_empty_needs_no_spark,
    test_outputs_validate_against_committed_schema,
    test_metrics_carries_no_verdicts_and_summary_no_numbers,
    test_per_mno_breakdown_in_addition_to_overall,
    test_per_mno_distincts_do_not_sum_to_overall,
    test_no_partition_by_means_no_per_mno_scan,
    test_idempotent_rerun_is_byte_identical,
    test_result_paths_are_canonical,
    test_config_rejects_bad_mode_and_missing_dataset,
    test_per_mno_allowed_values_needs_partition_by,
    test_validity_counts_non_null_format_mismatches,
    test_victim_exclusion_counts_wrong_and_null,
    test_currency_counts_rows_past_review_cycle,
]
_failed = 0
for _t in _tests:
    try:
        _t(); print("PASS  " + _t.__name__)
    except AssertionError as e:
        _failed += 1; print("FAIL  " + _t.__name__ + ": " + str(e))
    except Exception as e:
        _failed += 1; print("ERROR " + _t.__name__ + ": " + type(e).__name__ + ": " + str(e))
print()
print(str(len(_tests) - _failed) + "/" + str(len(_tests)) + " passed")


PASS  test_completeness_counts_nulls_per_column
PASS  test_completeness_only_required_columns_become_rules
PASS  test_completeness_matches_required_case_insensitively
PASS  test_completeness_missing_required_column_goes_red
PASS  test_consistency_flat_allowed_values
PASS  test_consistency_null_is_not_a_domain_violation
PASS  test_consistency_per_mno_matrix
PASS  test_uniqueness_duplicate_key_rate
PASS  test_uniqueness_null_key_parts_still_counted
PASS  test_uniqueness_no_primary_keys_means_full_row
PASS  test_timeliness_batch_future_event_fails
PASS  test_timeliness_batch_coverage_window_and_recency
PASS  test_timeliness_streaming_gap_and_delay_gate
PASS  test_timeliness_streaming_negative_gap_is_a_clock_bug
PASS  test_timeliness_absent_when_yaml_omits_it
PASS  test_empty_table_must_not_report_passed
PASS  test_empty_table_verdict_is_still_schema_valid
PASS  test_non_empty_table_has_no_empty_rule
PASS  test_build_summary_empty_needs_no_spark
PASS  test_outputs_validate_against_committe